In [1]:
import re
import pandas as pd

In [2]:
dictionary = {
    'hx': 'history',
    '#': 'fracture',
    'fx': 'fracture',
    'c/o': 'complaining of',
    'f/u': 'follow up',
    'fu': 'follow up',
    'w': 'with',
    'ortho': 'orthopaedics',
    'psych': 'psychiatry',
    'cardio': 'cardiology',
    'ophtho': 'ophthalmology',
    'ENT': 'otolaryngology',
    'plastics': 'plastic surgery',
    'physio': 'physiotherapy',
    'heme': 'hematology',
    'rx': 'prescription',
    'tx': 'treatment',
    #'x': 'times',
    'd': 'days ago',
    'mo': 'months',
    'lac': 'laceration',
    '@': 'at',
    'hpi': 'history of',
    'abx': 'antibiotics',
    'rom': 'range of motion',
    'fx': 'fracture',
    'prox': 'proximal',
    'wb': 'weight bear',
    'dist': 'distal',
    'min': 'miniutes',
    'mins': 'minutes',
    'sec': 'seconds',
    'secs': 'seconds',
    'hr': 'hours',
    'hrs': 'hours',
    'fb': 'foreign body',
    'hi': 'head injury',
    'approx': 'approximately',
    'sw': 'social worker',
    'sx': 'symptoms',
    'sh': 'self-harm',
    #'\?': 'possibly',
    'pt': 'patient',
    "pt's": "patient's",
    'r': 'right',
    'rt': 'right',
    'l': 'left',
    'lt': 'left',
    'ft': 'feet',
    'm': 'meters',
    'mm': 'millimeters',
    'cm': 'centimeters',
    'km/h': 'kilometers per hour',
    'kmph': 'kilometers per hour',
    'kph': 'kilometers per hour',
    'km': 'kilometers',
    'bsa': 'Body Surface Area',
    'tbsa': 'Total Body Surface Area',
    'ed': 'emergency department',
    'pip': 'proximal interphalangeal joint',
    'mt': 'metatarsal',
    'hep b': 'Hepatitis B',
    'plastics': 'plastic surgery',
    'loonie': '$1 coin',
    'toonie': '$2 coin',
    #'am': 'morning',
    'OCD': 'Obsessive-Compulsive Disorder',
    'ADHD': 'Attention Deficit Hyperactivity Disorder',
    'BPD': 'Borderline Personality Disorder',
    'PTSD': 'Post-Traumatic Stress Disorder',
    'MDD': 'Major Depressive Disorder',
    'GDD': 'Global Developmental Delay',
    'ASD': 'Autism Spectrum Disorder',
    'GAD': 'Generalized Anxiety Disorder',
    'ODD': 'Oppositional Defiant Disorder',
    'LD': 'learning disability',
    'DMDD': 'Disruptive Mood Dysregulation Disorder',
    'NAFLD': 'Non-alcoholic Fatty Liver Disease',
    'foosh': 'fall on an outstretched hand',
    'TMJ': 'Temporomandibular Joint',
    'MCP': 'metacarpophalangeal',
    'OD': 'overdose',
    'GnRH': 'Gonadotropin-Releasing Hormone',
    'trans': 'transgender',
    'WIC': 'walk-in clinic',
    'si': 'suicidal ideations',
    'LOC': 'altered level of consciousness',
    'CT': 'Computed Tomography',
    'TTC': 'Toronto Transit Commission',
    'FMD': 'family medical doctor',
    'PCP': 'primary care provider',
    'PMD': 'doctor of primary medicine',
    'MVC': 'motor vehicle collision',
    'EMS': 'Emergency Medical Services',
    'EKG': 'electrocardiogram',
    'ECG': 'electrocardiogram',
    'NBNB': 'non-bloody, non-bilious',
    'dw': 'discussed with',
    'RUQ': 'right upper quadrant'
    }



terms_to_replace = list(dictionary.keys())
replacements = list(dictionary.values())

term_mapping = {k.lower(): v for k, v in zip(terms_to_replace, replacements)}

patterns = [f"(?<![a-zA-Z0-9]){re.escape(k)}(?![a-zA-Z0-9])" for k in terms_to_replace]
pattern = re.compile("|".join(patterns), re.IGNORECASE)

In [3]:
def remove_extra_spaces(text):
    """
    Removes extra spaces from the given text.

    Args:
        text (str): The input text.

    Returns:
        str: The modified text without extra spaces.
    """
    words = text.split()
    new_text = ' '.join(words)
    return new_text

def capitalize_sentences(text):
    """
    Capitalizes the first letter of each sentence in the given text.

    Args:
        text (str): The input text.

    Returns:
        str: The modified text with capitalized sentences.
    """
    sentences = text.split('. ')
    capitalized_sentences = [sentence[0].upper() + sentence[1:] for sentence in sentences]
    return '. '.join(capitalized_sentences)

def push_punctuations(text):
    """
    Pushes punctuations to the previous word in the given text.

    Args:
        text (str): The input text.

    Returns:
        str: The modified text with punctuations pushed.
    """
    punctuations = [".", ",", ";", ":", "/", "?", "!", "\\"]
    counter = 0
    for i in range(len(text)):
        if text[i - counter] in punctuations and text[i - 1 - counter] == " ":
            text = text[:i - 1 - counter] + text[i - counter:]
            counter += 1
        elif text[i - counter] == " " and text[i - 1 - counter] == "/":
            text = text[:i - counter] + text[i - counter + 1:]
            counter += 1
    return text

def replace_terms(match):
    """
    Replaces the matched term with its corresponding value.

    Args:
        match (re.Match): The matched term.

    Returns:
        str: The replacement text.
    """
    match_text = match.group(0)
    term = re.search('|'.join(terms_to_replace), match_text, re.IGNORECASE)
    replacement = term_mapping[term.group(0).lower()]
    return match_text.replace(term.group(0), replacement)

def modify_text(text):
    """
    Modifies the text by replacing terms from the dictionary.

    Args:
        text (str): The input text.

    Returns:
        str: The modified text with replaced terms.
    """
    modified_text = re.sub(r'(?<=\d)(?=' + '|'.join(terms_to_replace) + r')', ' ', text)
    modified_text = pattern.sub(replace_terms, modified_text)
    return modified_text

def final(text):
    """
    Adds final touches to the text by capitalizing sentences, removing extra spaces, and adding a period at the end.

    Args:
        text (str): The input text.

    Returns:
        str: The final modified text.
    """
    new_text = capitalize_sentences(remove_extra_spaces(text))
    if new_text[-1] != '.':
        new_text = new_text + '.'
    return new_text


In [4]:
def summary_cleanup(text):
    """
    Cleans up the summary text by applying various modifications.

    Args:
        text (str): The input text.

    Returns:
        str: The cleaned up summary text.
    """
    new_text = capitalize_sentences(remove_extra_spaces(text))
    pushed_text = push_punctuations(new_text)
    replaced_text = modify_text(pushed_text)
    final_text = final(replaced_text)
    return final_text